In [24]:
%pip install openai sentence_transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
import re
from typing import List, Dict, Tuple
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer


In [ ]:
OPENAI_API_KEY       = os.getenv("OPENAI_API_KEY", "")
OPENAI_BASE_URL      = "http://localhost:1234/v1"
OPENAI_EMBED_MODEL = "text-embedding-3-small"
GEN_BACKEND   = "lmstudio" 
LMSTUDIO_CHAT_MODEL  = os.getenv("LMSTUDIO_CHAT_MODEL", "qwen/qwen3-30b-a3b_q4")


TOP_K                = int(os.getenv("TOP_K", "5")) # maximale Anzahl an Referenzen/Passagen 
ACCEPT_THRESHOLD     = float(os.getenv("ACCEPT_THRESHOLD", "0.30"))  # Akzeptanzschwellenwert 
EMBED_BACKEND = os.getenv("EMBED_BACKEND", "local")
MAX_WORDS_PER_CHUNK  = int(os.getenv("MAX_WORDS_PER_CHUNK", "120")) # Text wird in Chunks bzw. Passagen gesplittet 
OVERLAP_SENTENCES    = int(os.getenv("OVERLAP_SENTENCES", "2"))

In [27]:
def _openai_client():
    from openai import OpenAI
    if OPENAI_BASE_URL:
        return OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "not-needed")
    return OpenAI(api_key=OPENAI_API_KEY)

In [28]:
def split_sentences(text: str) -> List[str]:
    import re
    sents = re.split(r'(?<=[\.\!\?])\s+', text.strip())
    return [s for s in sents if s]

def chunk_text(text: str,
               max_words: int = MAX_WORDS_PER_CHUNK,
               overlap_sentences: int = OVERLAP_SENTENCES) -> List[Dict]:
    sents = split_sentences(text)
    chunks = []
    i = 0
    while i < len(sents):
        current = []
        n_words = 0
        start_i = i
        while i < len(sents) and (n_words + len(sents[i].split())) <= max_words:
            current.append(sents[i])
            n_words += len(sents[i].split())
            i += 1
        chunk = " ".join(current)
        chunks.append({"id": f"chunk-{len(chunks)+1}",
                       "text": chunk,
                       "start_sentence": start_i})
        # Satz-Overlap
        i = max(i - overlap_sentences, i)
        if i == start_i:  # falls Satz > max_words
            i += 1
    return chunks

In [29]:
class Embedder:
    def __init__(self, backend: str = "local"):
        self.backend = backend
        if backend == "local":
            try:
                from sentence_transformers import SentenceTransformer
            except ImportError as e:
                raise SystemExit("Bitte installiere sentence-transformers: pip install sentence-transformers") from e
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
        elif backend == "openai":
            if not OPENAI_API_KEY:
                raise SystemExit("Setze OPENAI_API_KEY für OPENAI-Embeddings.")
            self.client = _openai_client()
        else:
            raise ValueError("backend muss 'local' oder 'openai' sein.")

    def encode(self, texts: List[str]) -> np.ndarray:
        if self.backend == "local":
            vecs = self.model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
        else:
            resp = self.client.embeddings.create(model=OPENAI_EMBED_MODEL, input=texts)
            vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
            # L2-Normalisierung für Cosine als Skalarprodukt
            norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
            vecs = vecs / norms
        return vecs

In [30]:
def retrieve_top_k_with_threshold(chunks: List[Dict], chunk_vecs: np.ndarray,
                                  query_vec: np.ndarray, k: int = TOP_K,
                                  threshold: float = ACCEPT_THRESHOLD) -> Tuple[List[Tuple[int, float]], float, bool]:
    """
    Gibt Top-k (Index, Score), den genutzten Threshold und eine Bool-Entscheidung zurück.
    Entscheidung = True, wenn top1 >= threshold, sonst False.
    """
    scores = chunk_vecs @ query_vec  # Cosine (bei normalisierten Embeddings)
    idx_sorted = np.argsort(-scores)[:k]
    hits = [(int(i), float(scores[i])) for i in idx_sorted]
    top_ok = len(hits) > 0 and hits[0][1] >= threshold
    return hits, float(threshold), top_ok

In [ ]:
def build_prompt(selected_chunks: List[Dict], question: str) -> str:
    ctx_lines = []
    for i, ch in enumerate(selected_chunks, start=1):
        ctx_lines.append(f"[Quelle {i}]\n{ch['text']}")
    ctx = "\n\n".join(ctx_lines)
    prompt = f"""Du bist ein präziser Assistent. Beantworte die Frage AUSSCHLIESSLICH anhand des KONTEXTES.
Wenn Informationen fehlen, antworte exakt: "Keine Daten vorhanden."

KONTEXT:
{ctx}

FRAGE:
{question}

ANTWORT (knapp, auf Deutsch):
"""
    return prompt.strip()

def generate_answer(prompt: str) -> str:
    if GEN_BACKEND == "none":
        return "(LLM deaktiviert) Kontext bereitgestellt. Setze GEN_BACKEND=openai oder lmstudio."
    client = _openai_client()
    model_name = OPENAI_CHAT_MODEL if GEN_BACKEND == "openai" else LMSTUDIO_CHAT_MODEL
    resp = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

In [36]:
def demo():
    print("== RAG Demo mit Embeddings (fixer Threshold) ==")
    CORPUS_PATH="training_ki_supply_chain.txt"
    if CORPUS_PATH is not None and os.path.isfile(CORPUS_PATH):
        with open(CORPUS_PATH, "r", encoding="utf-8") as f:
            corpus = f.read()
        print(f"Lade Korpus aus Datei: {CORPUS_PATH} ({len(corpus.split())} Wörter)\n")
    else:
        corpus = make_demo_corpus(repeat=17)
        print(f"Länge des Korpus: {len(corpus.split())} Wörter\n")

    # 1) Chunking
    chunks = chunk_text(corpus, max_words=MAX_WORDS_PER_CHUNK, overlap_sentences=OVERLAP_SENTENCES)
    print(f"Erzeugte Chunks: {len(chunks)}  (max {MAX_WORDS_PER_CHUNK} Wörter, Overlap {OVERLAP_SENTENCES} Sätze)")

    # 2) Embeddings vorbereiten
    embedder = Embedder(backend=EMBED_BACKEND)
    chunk_vecs = embedder.encode([c["text"] for c in chunks])
    print(chunk_vecs)
    # 3) Beispiel-Fragen
    queries = [
        "Wozu eignen sich Batteriespeicher im Strommarkt?",
        "Was ist KI?",
        "Was bedeutet Vehicle-to-Grid?",
        "Was ist 1+1?",
        "Was ist 2+2?"
    ]

    for q in queries:
        print("\n" + "-" * 80)
        print("FRAGE:", q)
        q_vec = embedder.encode([q])[0]

        # 4) Retrieval + fester Threshold
        hits, thr, ok = retrieve_top_k_with_threshold(chunks, chunk_vecs, q_vec, k=TOP_K, threshold=ACCEPT_THRESHOLD)
        print(f"Top-{TOP_K} Treffer (Cosine-Similarity, Threshold={thr:.2f}):")
        for rank, (i, score) in enumerate(hits, start=1):
            preview = chunks[i]["text"][:180].replace("\n", " ")
            bar = "█" * int(max(1, round(score * 10)))  # simple Score-Bar
            print(f"{rank:>2}. score={score:.3f} {bar:<10}  |  {chunks[i]['id']}: {preview}...")

        if not ok:
            print("\n--- Entscheidung: NICHT relevant (Top-1 < Threshold) ---")
            print("Antwort: Keine Daten vorhanden.")
            continue

        # 5) Prompt + (6) optionale Generation
        selected = [chunks[i] for i, _ in hits]
        prompt = build_prompt(selected, q)
        print("\n#----- RAG Prompt (gekürzt) -----#")
        preview = prompt[:700] + ("..." if len(prompt) > 700 else "")
        print(preview)

        answer = generate_answer(prompt)
        print("\n#----- Antwort -----#")
        print(answer)

In [37]:
demo()

== RAG Demo mit Embeddings (fixer Threshold) ==
Lade Korpus aus Datei: training_ki_supply_chain.txt (17582 Wörter)

Erzeugte Chunks: 157  (max 120 Wörter, Overlap 2 Sätze)
[[-0.03354181  0.13104515 -0.02881807 ...  0.02576637 -0.03920124
  -0.01790407]
 [-0.11883843  0.04829877 -0.00254813 ...  0.12640955  0.04654917
  -0.01571717]
 [-0.08061561  0.02499443 -0.06839337 ... -0.00831254  0.01815724
   0.03455847]
 ...
 [-0.09547149  0.00379016 -0.04542872 ... -0.05773456 -0.00287474
  -0.01781656]
 [-0.01482351  0.00420326 -0.06261009 ...  0.00177428 -0.05208261
  -0.10407891]
 [-0.0782075   0.07367285 -0.01147301 ...  0.0012351  -0.10001688
  -0.00034674]]

--------------------------------------------------------------------------------
FRAGE: Wozu eignen sich Batteriespeicher im Strommarkt?
Top-5 Treffer (Cosine-Similarity, Threshold=0.30):
 1. score=0.331 ███         |  chunk-73: Prognose, an welchen Tagen wie viele Einlagerungen/Retouren anfallen, um Personal und Fläche optimal berei